# ECE661 LeJEPA: first baselines, all in one notebook

**Runtime → Change runtime type → GPU** (T4 works; L4/A100 are much faster), then **Runtime → Run all**.

What it does, top to bottom:
1. Mounts Google Drive. Checkpoints, logs, results and plots go to `DRIVE_DIR`.
2. Pulls the code from GitHub and installs it (including the official `lejepa` package).
3. Runs the unit tests (seconds).
4. **Cost probe:** 1 epoch per method, then prints estimated hours for `EPOCHS`.
5. Trains LeJEPA, SimCLR and SimSiam on CIFAR-10 (ResNet-18), each followed by the shared frozen-backbone evaluation: k-NN, linear probe with 1/10/100% labels, effective rank.
6. Writes a results table (CSV) and training-curve plots to Drive.

**If the runtime disconnects:** reconnect and *Run all* again. Finished runs are skipped and unfinished runs resume from their last checkpoint on Drive (saved every `SAVE_EVERY` epochs).

Paper references for CIFAR-10, ResNet-18, linear probe: LeJEPA 92.5% (Table 5), SimSiam 91.8% (800 epochs), SimCLR 91.1% (800 epochs; SimSiam paper, Appendix D). Fewer epochs will land lower.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/zianye183/ECE661-LeJEPA.git"
BRANCH = "baselines/paper-configs"      # switch to "main" once this branch is merged
DRIVE_DIR = "/content/drive/MyDrive/ECE661/lejepa"
WORKDIR = "/content/ECE661-LeJEPA"

METHODS = ["lejepa", "simclr", "simsiam"]
DATASET = "cifar10"
SEED = 0
EPOCHS = 100           # first baseline; the papers use 800. Check the cost probe estimate first.
SAVE_EVERY = 5         # checkpoint interval in epochs (resume granularity after a disconnect)
EXPERIMENT = f"baseline_e{EPOCHS}"
RUN_COST_PROBE = True

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime -> Change runtime type -> GPU"

## 2. Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR, RESULTS_DIR = f"{DRIVE_DIR}/runs", f"{DRIVE_DIR}/results"
os.makedirs(RUNS_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print("Saving to", DRIVE_DIR)

## 3. Code and dependencies

In [ ]:
if not os.path.exists(WORKDIR):
    !git clone -q -b {BRANCH} {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} checkout -q {BRANCH} && git -C {WORKDIR} pull -q --ff-only
%cd {WORKDIR}
!git log -1 --oneline
!pip install -q -e ".[dev]"
NUM_WORKERS = min(8, os.cpu_count() or 2)
COMMON = (f"--set data.dataset={DATASET} --set data.root=/content/data --set data.num_workers={NUM_WORKERS} "
          f"--runs-dir {RUNS_DIR} --results-dir {RESULTS_DIR}")
print("CPU workers:", NUM_WORKERS)

In [ ]:
!python -m pytest -q tests/test_sigreg.py tests/test_losses.py tests/test_eval.py

## 4. Cost probe
One epoch per method without evaluation. The time includes CUDA warm-up, so the estimate is
slightly pessimistic. If a method's total is too long for one Colab session, lower `EPOCHS`
(and rerun from Settings) or just let it resume across sessions.

In [ ]:
import json
if RUN_COST_PROBE:
    for m in METHODS:
        !python train.py --method {m} --seed {SEED} {COMMON} --set experiment=cost_probe --set train.epochs=1 --set train.warmup_epochs=0 --skip-eval 2>&1 | tail -n 3
    total = 0.0
    for m in METHODS:
        rec = [json.loads(l) for l in open(f"{RUNS_DIR}/cost_probe/{m}/{DATASET}/s{SEED}/metrics.jsonl")][-1]
        hours = rec["time_s"] * EPOCHS / 3600
        total += hours
        print(f"{m:8s} {rec['time_s']:6.1f} s/epoch  peak {rec.get('peak_mem_gb', '?')} GB  ->  ~{hours:.1f} h for {EPOCHS} epochs")
    print(f"all methods: ~{total:.1f} h")

## 5. Train and evaluate

In [ ]:
for m in METHODS:
    result_file = f"{RESULTS_DIR}/{EXPERIMENT}/{m}/{DATASET}/s{SEED}.json"
    if os.path.exists(result_file):
        print(f"{m}: done already, skipping ({result_file})")
        continue
    print(f"===== {m} =====")
    !python train.py --method {m} --seed {SEED} {COMMON} --set experiment={EXPERIMENT} --set train.epochs={EPOCHS} --set train.save_every={SAVE_EVERY} --resume

## 6. Results

In [ ]:
!python scripts/collect_results.py {RESULTS_DIR}/{EXPERIMENT} --out {DRIVE_DIR}/{EXPERIMENT}_summary.csv

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m in METHODS:
    path = f"{RUNS_DIR}/{EXPERIMENT}/{m}/{DATASET}/s{SEED}/metrics.jsonl"
    if not os.path.exists(path):
        continue
    recs = [json.loads(l) for l in open(path)]
    ep = [r["epoch"] for r in recs]
    axes[0].plot(ep, [r["loss"] for r in recs], label=m)
    axes[1].plot(ep, [r.get("probe_acc", float("nan")) for r in recs], label=m)
axes[0].set(title="SSL training loss (scales differ by method)", xlabel="epoch", yscale="log")
axes[1].set(title="Online linear probe accuracy (train batches)", xlabel="epoch", ylabel="accuracy")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/{EXPERIMENT}_curves.png", dpi=150)
plt.show()
print("Saved", f"{DRIVE_DIR}/{EXPERIMENT}_curves.png")